# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
!{sys.executable} scripts/run_all.py



▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page needs review if it's stale-but-visible, declining-with-demand,
thin-but-visible, slipping-on-page-1, or getting impressions but not clicks.

Reason codes:
- stale_visible_page: not updated in 180+ days AND 500+ impressions
- declining_with_demand: trend_direction == "down" AND 100+ impressions  
- thin_visible_page: word_count < 1200 AND 250+ impressions
- page_one_decay_risk: avg_position <= 10 AND page is 180+ days old
- low_ctr_visible_page: 500+ impressions, position 1-20, but ctr < 0.5
- general_refresh_review: fallback when none of the above fire

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# find repo root by walking up until we find the data folder
p = Path.cwd()
while not (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    p = p.parent
ROOT = p

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

# score: normalize impressions (visibility) + staleness, weight them
def pct_rank(s):
    return s.rank(pct=True)

df["visibility_score"] = pct_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = pct_rank(df["days_since_last_update"])
df["baseline_score"] = (0.5 * df["visibility_score"] + 0.5 * df["freshness_risk_score"]).clip(0, 1)

df["rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)
out = df.sort_values("rank")[["content_id", "client_id", "rank", "baseline_score",
                                "reason_codes", "impressions_90d", "days_since_last_update",
                                "avg_position", "ctr", "trend_direction", "is_declining_label"
                                if "is_declining_label" in df.columns else "trend_direction"]]

out_path = ROOT / "work" / "outputs" / "baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)
print(f"Wrote {len(out)} rows to {out_path}")
out.head(20)

Wrote 30000 rows to /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv


,content_id,client_id,rank,baseline_score,reason_codes,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,1,0.991317,stale_visible_page|declining_with_demand|low_c...,61678,194,19.7,0.15,down,down
10870,content_a5dbb404bdc2,client_f369cb89fc,2,0.991200,low_ctr_visible_page,79035,106,8.7,0.07,stable,stable
16514,content_7368877ea310,client_7f2253d7e2,3,0.991117,stale_visible_page|declining_with_demand,59472,194,24.8,0.13,down,down
6345,content_47b8b12d581e,client_7f2253d7e2,4,0.983933,declining_with_demand,40305,106,28.4,0.96,down,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,5,0.975817,stale_visible_page|declining_with_demand,25715,194,22.2,0.23,down,down
16648,content_69fad7e6c50c,client_7f2253d7e2,6,0.975650,declining_with_demand,28000,106,4.7,1.32,down,down
12400,content_482aff19e9cc,client_7f2253d7e2,7,0.974017,declining_with_demand,26287,106,13.2,2.43,down,down
22197,content_6ac3ab740bbf,client_f369cb89fc,8,0.969858,declining_with_demand|low_ctr_visible_page,22462,106,4.6,0.14,down,down
8006,content_cb7e312f5d32,client_9f14025af0,9,0.969033,thin_visible_page,21272,151,12.6,2.45,up,up
27096,content_ac1d924c6a70,client_7f2253d7e2,10,0.968683,declining_with_demand,21853,106,31.7,0.13,down,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = out.head(20)
top20

,content_id,client_id,rank,baseline_score,reason_codes,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,1,0.991317,stale_visible_page|declining_with_demand|low_c...,61678,194,19.7,0.15,down,down
10870,content_a5dbb404bdc2,client_f369cb89fc,2,0.991200,low_ctr_visible_page,79035,106,8.7,0.07,stable,stable
16514,content_7368877ea310,client_7f2253d7e2,3,0.991117,stale_visible_page|declining_with_demand,59472,194,24.8,0.13,down,down
6345,content_47b8b12d581e,client_7f2253d7e2,4,0.983933,declining_with_demand,40305,106,28.4,0.96,down,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,5,0.975817,stale_visible_page|declining_with_demand,25715,194,22.2,0.23,down,down
16648,content_69fad7e6c50c,client_7f2253d7e2,6,0.975650,declining_with_demand,28000,106,4.7,1.32,down,down
12400,content_482aff19e9cc,client_7f2253d7e2,7,0.974017,declining_with_demand,26287,106,13.2,2.43,down,down
22197,content_6ac3ab740bbf,client_f369cb89fc,8,0.969858,declining_with_demand|low_ctr_visible_page,22462,106,4.6,0.14,down,down
8006,content_cb7e312f5d32,client_9f14025af0,9,0.969033,thin_visible_page,21272,151,12.6,2.45,up,up
27096,content_ac1d924c6a70,client_7f2253d7e2,10,0.968683,declining_with_demand,21853,106,31.7,0.13,down,down


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
used_columns = ["impressions_90d", "days_since_last_update"]  # whatever you scored on
leaky = [c for c in used_columns if c in ("trend_pct", "health_score", "priority_score", "action_type")]
print("Leaky columns used:", leaky)  # should print []

Leaky columns used: []


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.